In [1]:
print('Start')

Start


In [2]:
# pip install tensorflow

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Concatenate, Dropout
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [4]:
def check_dataset(image_dir, csv_path, id_column):
    print(f"\n Checking dataset:\nImages: {image_dir}\nCSV: {csv_path}\n")
    
    # Load CSV
    df = pd.read_csv(csv_path)
    
    # Get images in folder
    images_in_folder = [f for f in os.listdir(image_dir) if os.path.isfile(os.path.join(image_dir, f))]
    img_names_no_ext = [os.path.splitext(f)[0] for f in images_in_folder]
    
    # Get IDs from CSV
    csv_ids = df[id_column].astype(str).tolist()
    
    # Images without matching CSV
    missing_in_csv = [img for img in img_names_no_ext if img not in csv_ids]
    
    # CSV rows without matching image file
    missing_in_folder = [id_val for id_val in csv_ids if id_val not in img_names_no_ext]
    
    # Results
    print(f"Total images in folder: {len(images_in_folder)}")
    print(f"Total rows in CSV: {len(df)}")
    print(f"Images without matching CSV row: {len(missing_in_csv)}")
    print(f"CSV rows without matching image file: {len(missing_in_folder)}")
    
    if missing_in_csv:
        print("\nImages missing in CSV:")
        for m in missing_in_csv:
            print(m)
    else:
        print("\n All images have matching rows in CSV.")
    
    if missing_in_folder:
        print("\nCSV rows missing image file:")
        for m in missing_in_folder:
            print(m)
    else:
        print("\n All CSV rows have matching image files.")
        
# Validation dataset check
check_dataset(
    r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Validation\images",
    r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Validation\sliders_input.csv",
    id_column="id_global"
)

# Train dataset check
check_dataset(
    r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Train\images",
    r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Train\sliders.csv",
    id_column="id_global"
)


 Checking dataset:
Images: C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Validation\images
CSV: C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Validation\sliders_input.csv

Total images in folder: 493
Total rows in CSV: 493
Images without matching CSV row: 0
CSV rows without matching image file: 0

 All images have matching rows in CSV.

 All CSV rows have matching image files.

 Checking dataset:
Images: C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Train\images
CSV: C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Train\sliders.csv

Total images in folder: 2538
Total rows in CSV: 2538
Images without matching CSV row: 0
CSV rows without matching image file: 0

 All images have matching rows in CSV.

 All CSV rows have matching image files.


In [5]:
## remove extra images

In [6]:
train_img_dir = r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Train\images"
train_csv_path = r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Train\sliders.csv"
val_img_dir = r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Validation\images"
val_csv_path = r"C:\Users\Krishan\Juyp\Aftershoot\Dataset\dataset\Validation\sliders_input.csv"



In [7]:
train_df = pd.read_csv(train_csv_path)
val_df = pd.read_csv(val_csv_path)

In [8]:
train_df.head()


,copyCreationTime,captureTime,touchTime,id_global,grayscale,aperture,flashFired,focalLength,isoSpeedRating,shutterSpeed,Temperature,Tint,currTemp,currTint
0,-63113817600,2024-01-14T16:56:50.67,741426608.1,C68C8010-495C-4427-9F4D-664C2D71EFAD,0,2.970854,1,100.0,1250,7.965784,4150,2,6317,4
1,-63113817600,2023-05-28T20:08:51.87,741426607.1,8EFC0EC0-0936-41CC-81BD-513B35D2CB23,0,7.614710,1,24.0,500,3.000000,4700,4,5767,13
2,-63113817600,2023-06-09T20:54:30.13,741426607.2,4A28220F-024E-4637-80ED-B4533578AFEB,0,3.614710,1,125.0,1000,7.321928,5000,12,5496,6
3,-63113817600,2022-06-10T13:02:12.98,741426606.5,05A76E40-9B2C-40FD-95D4-EF976598640C,0,2.970854,0,40.0,320,7.965784,3150,9,3730,12
4,-63113817600,2023-09-30T17:08:05,741426607.8,B63A179E-232C-4133-BB24-8784B60DECEE,0,2.970854,0,55.0,800,7.643856,3633,4,3661,-6


In [9]:
val_df.head()

,copyCreationTime,captureTime,id_global,grayscale,hasDevelopAdjustmentsEx,aperture,flashFired,focalLength,isoSpeedRating,shutterSpeed,currTemp,currTint
0,-63113817600,2025-09-27T16:14:53,EB5BEE31-8D4F-450A-8BDD-27C762C75AA6,0,1,4.0,0,21.2,800,4.906891,6613,14
1,-63113817600,2025-09-27T16:14:57,DE666E1F-0433-4958-AEC0-9A0CC0F81036,0,1,4.0,0,16.0,800,4.906891,6613,14
2,-63113817600,2025-09-27T16:15:17,F6A6EA9C-A5C2-4BBA-9812-5CE52B818CB6,0,1,4.0,0,52.4,800,4.906891,6613,14
3,-63113817600,2025-09-27T16:16:11,BCC39DEF-598C-491A-A3CA-14A249717F36,0,1,4.0,0,16.0,800,4.906891,6782,14
4,-63113817600,2025-09-27T16:16:26,390ED94E-0066-4822-99B9-8F1568BDFBF5,0,1,4.0,0,54.5,800,4.906891,6782,14


In [10]:
meta_features = ['grayscale', 'aperture', 'flashFired', 'focalLength', 
                 'isoSpeedRating', 'shutterSpeed', 'currTemp', 'currTint']

X_meta = train_df[meta_features].values
y_temp = train_df['Temperature'].values
y_tint = train_df['Tint'].values

In [11]:
scaler = StandardScaler()
X_meta_scaled = scaler.fit_transform(X_meta)


In [12]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def load_images(image_ids, img_dir, target_size=(256, 256)):
    images = []
    for img_id in image_ids:
        img_path = os.path.join(img_dir, img_id + ".tif")  # TIFF extension
        if not os.path.exists(img_path):
            print(f"⚠ Warning: Image not found - {img_path}")
            continue
        img = load_img(img_path, target_size=target_size)
        img_array = img_to_array(img) / 255.0  # normalize
        images.append(img_array)
    return np.array(images, dtype=np.float32)

In [13]:
X_images = load_images(train_df['id_global'], train_img_dir)

In [14]:
X_img_train, X_img_val, X_meta_train, X_meta_val, y_temp_train, y_temp_val, y_tint_train, y_tint_val = train_test_split(
    X_images, X_meta_scaled, y_temp, y_tint, test_size=0.2, random_state=42
)

In [15]:

#  BUILD HYBRID MODEL

# Image branch
img_input = Input(shape=(256, 256, 3), name="image_input")
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=img_input)
x_img = Flatten()(base_model.output)
x_img = Dense(256, activation='relu')(x_img)
x_img = Dropout(0.3)(x_img)

# Metadata branch
meta_input = Input(shape=(len(meta_features),), name="meta_input")
x_meta = Dense(64, activation='relu')(meta_input)
x_meta = Dense(32, activation='relu')(x_meta)

# Fusion
x = Concatenate()([x_img, x_meta])
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

# Two outputs
temp_output = Dense(1, name="Temperature")(x)
tint_output = Dense(1, name="Tint")(x)

model = Model(inputs=[img_input, meta_input], outputs=[temp_output, tint_output])

model.compile(optimizer='adam', loss='mae')

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)      │ (None, 256, 256, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling (Rescaling)         │ (None, 256, 256, 3)       │               0 │ image_input[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ normalization (Normalization) │ (None, 256, 256, 3)       │               7 │ rescaling[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_1 (Rescaling)       │ (None, 256, 256, 3)       │               0 │ normalization[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv_pad (ZeroPadding2D) │ (None, 257, 257, 3)       │               0 │ rescaling_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv (Conv2D)            │ (None, 128, 128, 32)      │             864 │ stem_conv_pad[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_bn (BatchNormalization)  │ (None, 128, 128, 32)      │             128 │ stem_conv[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_activation (Activation)  │ (None, 128, 128, 32)      │               0 │ stem_bn[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_dwconv                │ (None, 128, 128, 32)      │             288 │ stem_activation[0][0]      │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_bn                    │ (None, 128, 128, 32)      │             128 │ block1a_dwconv[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_activation            │ (None, 128, 128, 32)      │               0 │ block1a_bn[0][0]           │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_squeeze            │ (None, 32)                │               0 │ block1a_activation[0][0]   │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reshape (Reshape)  │ (None, 1, 1, 32)          │               0 │ block1a_se_squeeze[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reduce (Conv2D)    │ (None, 1, 1, 8)           │             264 │ block1a_se_reshape[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_expand (Conv2D)    │ (None, 1, 1, 32)          │             288 │ block1a_se_reduce[0][0]    │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 25,061,253 (95.60 MB)

 Trainable params: 25,019,230 (95.44 MB)

 Non-trainable params: 42,023 (164.16 KB)

In [16]:

# -------------------------------
# 8. TRAIN MODEL
# -------------------------------
history = model.fit(
    [X_img_train, X_meta_train],
    [y_temp_train, y_tint_train],
    validation_data=([X_img_val, X_meta_val], [y_temp_val, y_tint_val]),
    epochs=10,
    batch_size=16
)


Epoch 1/10


C:\Users\Krishan\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['image_input', 'meta_input']. Received: the structure of inputs=('*', '*')
  warnings.warn(


127/127 ━━━━━━━━━━━━━━━━━━━━ 850s 6s/step - Temperature_loss: 2083.5874 - Tint_loss: 227.2812 - loss: 2310.8730 - val_Temperature_loss: 995.6306 - val_Tint_loss: 32.0088 - val_loss: 1029.0605
Epoch 2/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 725s 6s/step - Temperature_loss: 949.6393 - Tint_loss: 175.9041 - loss: 1125.5437 - val_Temperature_loss: 998.7767 - val_Tint_loss: 67.2656 - val_loss: 1064.7148
Epoch 3/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 727s 6s/step - Temperature_loss: 867.2938 - Tint_loss: 100.4076 - loss: 967.7017 - val_Temperature_loss: 1119.5249 - val_Tint_loss: 36.3817 - val_loss: 1157.6254
Epoch 4/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 750s 6s/step - Temperature_loss: 803.1130 - Tint_loss: 59.4242 - loss: 862.5363 - val_Temperature_loss: 2270.3997 - val_Tint_loss: 8.2482 - val_loss: 2280.4973
Epoch 5/10
127/127 ━━━━━━━━━━━━━━━━━━━━ 732s 6s/step - Temperature_loss: 700.8265 - Tint_loss: 32.8865 - loss: 733.7085 - val_Temperature_loss: 3534.5596 - val_Tint_loss: 11.4222 - val_loss: 3546.0239
Epo

In [17]:

# -------------------------------
# 9. LOAD VALIDATION IMAGES & META
# -------------------------------
X_val_images = load_images(val_df['id_global'], val_img_dir)
X_val_meta = scaler.transform(val_df[meta_features].values)


In [18]:

# -------------------------------
# 10. PREDICT
# -------------------------------
val_temp_preds, val_tint_preds = model.predict([X_val_images, X_val_meta])
val_temp_preds = np.round(val_temp_preds.flatten()).astype(int)
val_tint_preds = np.round(val_tint_preds.flatten()).astype(int)


C:\Users\Krishan\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['image_input', 'meta_input']. Received: the structure of inputs=('*', '*')
  warnings.warn(


16/16 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step


In [19]:

# -------------------------------
# 11. SAVE SUBMISSION
# -------------------------------
submission = pd.DataFrame({
    'id_global': val_df['id_global'],
    'Temperature': val_temp_preds,
    'Tint': val_tint_preds
})
submission.to_csv("submission1.csv", index=False)
print(" Submission saved as submission.csv")

 Submission saved as submission.csv


In [20]:
# -------------------------------
# Evaluate model performance after training
# -------------------------------

from sklearn import metrics
import pandas as pd
import numpy as np

# Get model predictions
pred_temp, pred_tint = model.predict([X_img_val, X_meta_val])

# Convert to DataFrame for clarity
actual = pd.DataFrame({
    'Temperature': y_temp_val.flatten(),
    'Tint': y_tint_val.flatten()
})

predicted = pd.DataFrame({
    'Temperature': pred_temp.flatten(),
    'Tint': pred_tint.flatten()
})

# Compute MAE
mae_temp = metrics.mean_absolute_error(actual['Temperature'], predicted['Temperature'])
mae_tint = metrics.mean_absolute_error(actual['Tint'], predicted['Tint'])

# Convert to normalized (1 / (1 + MAE)) score
mae_temperature_score = 1 / (1 + mae_temp)
mae_tint_score = 1 / (1 + mae_tint)

# Display results
print("\n Model Evaluation Results:")
print(f"MAE (Temperature): {mae_temp:.4f}  →  Score: {mae_temperature_score:.4f}")
print(f"MAE (Tint):        {mae_tint:.4f}  →  Score: {mae_tint_score:.4f}")

# Combined score for overall performance
combined_score = (mae_temperature_score + mae_tint_score) / 2
print(f"\n Overall Model Score: {combined_score:.4f}")


16/16 ━━━━━━━━━━━━━━━━━━━━ 30s 2s/step

 Model Evaluation Results:
MAE (Temperature): 1016.4797  →  Score: 0.0010
MAE (Tint):        7.3418  →  Score: 0.1199

 Overall Model Score: 0.0604


In [21]:
# Save entire model (architecture + weights + optimizer state)
model.save("combined_model.h5")
print("Model saved successfully as 'combined_model.h5'")

Model saved successfully as 'combined_model.h5'
